# **Example use of uORF-Predictor to obtain the matrix for Machine Learning**

This is a document showing how to use uORF-Predictor to extract all the features written in the code and obtain the matrix to train the model.

## *Pre-requisites*

### GTF file

To start with, a GTF file is needed to get the transcripts with the 5'UTR regions of each one. 
It should look something similar to this:

In [2]:
import os

gtf_file_path = os.path.join("..", "tests", "data", "Homo.sapiens.GRCh38_sample_chr22.gtf")

with open(file=gtf_file_path, mode="r", encoding='utf-8') as file:
    lines = file.readlines()
    for line in lines[160:169]:
        print(line.strip())

chr22	HAVANA	CDS	44732251	44732391	.	+	0	gene_id "ENSG00000186654.22"; transcript_id "ENST00000432186.6"; gene_type "protein_coding"; gene_name "PRR5"; transcript_type "protein_coding"; transcript_name "PRR5-206"; exon_number 7; exon_id "ENSE00003658491.1"; level 2; protein_id "ENSP00000400925.2"; transcript_support_level "2"; hgnc_id "HGNC:31682"; tag "alternative_5_UTR"; tag "basic"; tag "appris_alternative_1"; tag "CCDS"; ccdsid "CCDS14059.1"; havana_gene "OTTHUMG00000150460.6"; havana_transcript "OTTHUMT00000318206.2";
chr22	HAVANA	exon	44735027	44735162	.	+	.	gene_id "ENSG00000186654.22"; transcript_id "ENST00000432186.6"; gene_type "protein_coding"; gene_name "PRR5"; transcript_type "protein_coding"; transcript_name "PRR5-206"; exon_number 8; exon_id "ENSE00003692865.1"; level 2; protein_id "ENSP00000400925.2"; transcript_support_level "2"; hgnc_id "HGNC:31682"; tag "alternative_5_UTR"; tag "basic"; tag "appris_alternative_1"; tag "CCDS"; ccdsid "CCDS14059.1"; havana_gene "OTTHUM

As it can be seen in the file, the transcript `ENST00000432186.6` has three UTR regions, but not all of them correspond to the 5'UTR.

### uORFs file

In this example, a file with a few examples of uORFs will be used. The file has this shape:

In [3]:
import os
import pandas as pd

uorf_fpath = os.path.join("..", "tests", "data", "uorfs_example.csv")

uorf_df = pd.read_csv(uorf_fpath, sep=",", header=0)
print(uorf_df)

             tx_id     start       end start_codon
0  ENST00000649746  17256523  17256603         ATG


## *Usage*

The first mandatory step is to obtain all the 5'UTR regions in the `TranscriptCoordinates` form. 
Firstly, the GTF file must be converted into a `DataFrame` with the `gtf_to_dataframe()` method, this is done automatically by the class:

In [4]:
from uorf_predictor.gtf_io import GTFio

gft_df = GTFio(fpath= gtf_file_path).gtf_to_dataframe()
gft_df.head()

,seqname,feature,start,end,strand,transcript_id,gene_name
0,chr22,transcript,17178790,17258235,-,ENST00000649746.2,ADA2
1,chr22,exon,17258164,17258235,-,ENST00000649746.2,ADA2
2,chr22,exon,17256268,17256878,-,ENST00000649746.2,ADA2
3,chr22,exon,17221727,17222101,-,ENST00000649746.2,ADA2
4,chr22,exon,17209356,17209723,-,ENST00000649746.2,ADA2


Then, once the `DataFrame` is created, each 5'UTR region is transformed into a `TranscriptCoordinates`, the Reference genome used is *GRCh38/hg38*:

In [5]:
from uorf_predictor.gtf_io import GTFio
from gpsea.model.genome import GRCh38

transcripts = GTFio(fpath=gtf_file_path).extract_five_utrs(genome_build=GRCh38)
print(transcripts)

[TranscriptCoordinates(tx_id=ENST00000366425.4, five_utr=FiveUTRCoordinates(regions=1 regions: (chr22, 19723538, 19723569, +))), TranscriptCoordinates(tx_id=ENST00000432186.6, five_utr=FiveUTRCoordinates(regions=2 regions: (chr22, 44668712, 44668805, +), (chr22, 44702491, 44702501, +))), TranscriptCoordinates(tx_id=ENST00000623063.3, five_utr=FiveUTRCoordinates(regions=1 regions: (chr22, 40346499, 40346558, +))), TranscriptCoordinates(tx_id=ENST00000642974.1, five_utr=FiveUTRCoordinates(regions=2 regions: (chr22, 31753897, 31753996, +), (chr22, 31754861, 31754921, +))), TranscriptCoordinates(tx_id=ENST00000649746.2, five_utr=FiveUTRCoordinates(regions=4 regions: (chr22, 33560233, 33560305, -), (chr22, 33561590, 33562201, -), (chr22, 33596367, 33596742, -), (chr22, 33608745, 33608791, -))), TranscriptCoordinates(tx_id=ENST00000703965.1, five_utr=FiveUTRCoordinates(regions=2 regions: (chr22, 26837999, 26838057, -), (chr22, 26841401, 26841576, -)))]


After this, each line in the uORF file will be read, processed to get the features and put into a CSV file.

In [6]:
import os
import csv

from gpsea.model.genome import GRCh38, GenomeBuild, Strand
from uorf_predictor.uorf_extractor import fetch_cdna_from_ensembl, get_five_prime_sequence, obtain_uorf_in_five_utr, check_start_and_stop_codon
from uorf_predictor.uorf_features import gc_content, kozak_sequence_strength, cap_five_to_uorf_distance, intercistronic_distance, codon_count, Codon_features
from uorf_predictor.rna_features import RNA_folding

uorf_fpath = os.path.join("..", "tests", "data", "uorfs_example.csv")
output_fpath = os.path.join("..", "tests", "data", "example_results.csv")

header = ["tx_id", "length", "gc", "overlapping", "kozak_strength",
          "distance_to_cap", "intercistronic_distance", "cai", "codon_freq_std", "num_codon",
          "mfe_five", "mfe_start", "mfe_uorf", "atg_unpaired", "atg_num_unpaired",
          "unpaired_percentage", "shannon_entropy", "ubox_sum", "ensemble_diversity", "start_codon",
          "phylop", "phastcons"]

# Create CSV file and write header
with open(output_fpath, "w", newline="") as file:
    writer = csv.writer(file, delimiter=",")
    writer.writerow(header)

for _, row in uorf_df.iterrows():
    tx_id = row["tx_id"]
    tx_found = None

    # Search the transcript ID of the uORF in the trasncripts of the GTF file
    for tx in transcripts:
        clean_tx_id = tx.tx_id.split(".")[0]
        if clean_tx_id == tx_id:
            tx_found = tx
            break
    
    assert tx_found is not None, "Transcript not found"

    # Get the 5'UTR cDNA sequence
    tx_cdna_sequence = fetch_cdna_from_ensembl(transcript_id=tx_found.tx_id.split(".")[0])
    five_utr_cdna_sequence = get_five_prime_sequence(cdna_sequence=tx_cdna_sequence, five_utrs=tx_found.five_utr)
    
    results_tx = []

    tx_id_results = tx_id + "-" + str(row["start"]) + "-" + str(row["end"])

    results_tx.append(tx_id_results) # tx_id
    results_tx.append(row["end"] - row["start"] + 1) # Length (add one because the start is 0-based)

    uorf = obtain_uorf_in_five_utr(
        five_utrs=tx_found.five_utr,
        start_uorf=row["start"],
        end_uorf=row["end"],
    )   
    results_tx.append(gc_content(five_sequence=five_utr_cdna_sequence, uorf=uorf))
    results_tx.append(uorf.ouorf) # Overlapping
    results_tx.append(kozak_sequence_strength(five_sequence=five_utr_cdna_sequence, uorf=uorf)) # Kozak sequence strength
    results_tx.append(cap_five_to_uorf_distance(uorf=uorf)) # Distance to 5'cap
    results_tx.append(intercistronic_distance(five_sequence=five_utr_cdna_sequence, uorf=uorf)) # Intercistronic distance
    uorf_sequence = five_utr_cdna_sequence[uorf.uorf.start:uorf.uorf.end] # Always check if the uORF is given 0 or 1-based
    is_uorf = check_start_and_stop_codon(uorf_sequence=uorf_sequence)
    if is_uorf == True:
        codon_instance = Codon_features(uorf_sequence=uorf_sequence)
        results_tx.append(codon_instance.codon_adaptation_index()) # Codon Adaptation Index
        results_tx.append(codon_instance.codon_frequency_std()) # Codon frequency std
    results_tx.append(codon_count(uorf_sequence=uorf_sequence)) # Number of codons

    rna_instance = RNA_folding(five_utr_sequence=five_utr_cdna_sequence, uorf=uorf)
    results_tx.append(rna_instance.five_sequence_mfe()) # 5'UTR MFE
    results_tx.append(rna_instance.uorf_start_context_mfe()) # Start codon context MFE
    results_tx.append(rna_instance.uorf_sequence_mfe()) # uORF MFE
    results_tx.append(rna_instance.start_codon_is_unpaired()) # ATG is unpaired?
    results_tx.append(rna_instance.start_codon_number_unpaired()) # Number of nucleotides unpaired in ATG
    results_tx.append(rna_instance.uorf_unpaired_bases_percentage()) # Percentage of unpaired bases in uORF
    results_tx.append(rna_instance.shannon_entropy()) # Shannon entropy of uORF
    results_tx.append(rna_instance.ubox_total_probs_sum()) # Sum of all ubox probabilities
    results_tx.append(rna_instance.uorf_ensemble_diversity()) # uORF ensemble diversity
    results_tx.append(row["start_codon"]) # Start codon
    results_tx.append(0.01685) # PhyloP score (internally calculated)
    results_tx.append(0.001706) # PhastCons score (internally calculated)

    # Write results in csv
    with open(output_fpath, "a", newline='') as file:
        writer = csv.writer(file, delimiter=',')
        writer.writerow(results_tx)

Finally, you should get a file with lines like this one:

In [7]:
import csv

with open(output_fpath, mode="r", encoding="utf-8") as file:
    reader = csv.reader(file)
    
    for row in reader:
        print(row)


['tx_id', 'length', 'gc', 'overlapping', 'kozak_strength', 'distance_to_cap', 'intercistronic_distance', 'cai', 'codon_freq_std', 'num_codon', 'mfe_five', 'mfe_start', 'mfe_uorf', 'atg_unpaired', 'atg_num_unpaired', 'unpaired_percentage', 'shannon_entropy', 'ubox_sum', 'ensemble_diversity', 'start_codon', 'phylop', 'phastcons']
['ENST00000649746-17256523-17256603', '81', '0.43209876543209874', 'False', '1', '347', '676', '0.7068355232571196', '0.2990234960349921', '27.0', '-381.70001220703125', '-19.899999618530273', '-10.100000381469727', 'False', '1', '72.8395061728395', '5.0810565157223255', '11.908910205428606', '8.560255510427712', 'ATG', '0.01685', '0.001706']
